# Restaurant Tips: Loading, Exploring, and Filtering Data

In this notebook, you will investigate restaurant bills and tips recorded in `Data/tips.csv`. Begin with simple inspection tasks, then work up to conditional filtering, creating columns, and comparing groups.

**How to use this worksheet:** Read each question, write your attempt in the empty code cell, and then compare it with the worked solution in the code cell immediately below. The solutions are included for study; try each problem before running its answer.

## About the dataset

The shared teaching copy contains **244 restaurant bills** and these columns:

| Column | Meaning |
| --- | --- |
| `total_bill` | Total bill amount |
| `tip` | Tip amount |
| `sex` | Recorded sex of the person paying |
| `smoker` | Whether the person paying is recorded as a smoker |
| `day` | Day of the week |
| `time` | Meal period (`Lunch` or `Dinner`) |
| `size` | Number of people in the party |
| `price_per_person` | Per-person amount included in the source file |

The original source also included payer and payment identifiers. Those fields are deliberately omitted from the teaching copy because they are not needed for these exercises.

The exercises focus on bill, tip, meal, and party-size data. The observations describe this sample; they do not by themselves establish that one factor caused another.

## Exercise 1 — Load the CSV (Starter)

Import pandas and load `Data/tips.csv` into a DataFrame named `tips`. What is the shape of the loaded data? Use a path relative to this notebook.

In [ ]:
"""Load the restaurant records and report the table dimensions."""
from pathlib import Path

import pandas as pd

# Support a kernel started in either this Pandas folder or the repository root.
data_path = Path("Data/tips.csv")
if not data_path.exists():
    data_path = Path("Pandas") / "Data" / "tips.csv"
tips = pd.read_csv(data_path)

# shape is a tuple containing (number of rows, number of columns).
print("Dataset shape (rows, columns):", tips.shape)

## Exercise 2 — Preview the first and last records (Beginner)

Display the first five rows and the last five rows. Why are `head()` and `tail()` useful before analyzing a dataset?

## Exercise 3 — Inspect structure and numeric summaries (Beginner)

Use `info()` to report column names, non-null counts, and data types. Then use `describe()` to summarize `total_bill`, `tip`, `size`, and `price_per_person`. What are the median total bill and the largest tip? The file also contains a numeric card-number identifier; do not treat that identifier as a meaningful measurement.

In [ ]:
"""Inspect column types, completeness, and numeric distributions."""
# info() prints a compact structural summary, including non-null values and dtypes.
tips.info()

# Summarize restaurant measures only; the numeric card identifier is not a measurement.
restaurant_measures = ["total_bill", "tip", "size", "price_per_person"]
display(tips[restaurant_measures].describe())

# Read the median bill and the maximum recorded tip from the summary.
print("Median total bill:", tips["total_bill"].median())
print("Largest tip:", tips["tip"].max())

## Exercise 4 — Filter by meal period (Beginner)

Create a DataFrame containing only `Dinner` records. Display the first five matching rows and report how many dinner records there are.

In [ ]:
"""Select dinner visits with a Boolean condition."""
# Comparing a column with a value makes a True/False mask for each row.
dinner = tips.loc[tips["time"] == "Dinner"]
display(dinner.head())
print("Number of dinner records:", len(dinner))

## Exercise 5 — Combine conditions (Beginner to intermediate)

Find records for **non-smokers dining on Sunday**. Display the `day`, `smoker`, `total_bill`, and `tip` columns, and report the number of matching records. Remember to put each comparison in parentheses and use `&` for AND.

In [ ]:
"""Filter for Sunday visits where the recorded smoker value is No."""
# Use & for element-wise AND and parentheses around each comparison.
sunday_non_smokers = tips.loc[
    (tips["day"] == "Sun") & (tips["smoker"] == "No"),
    ["day", "smoker", "total_bill", "tip"],
]
display(sunday_non_smokers)
print("Matching records:", len(sunday_non_smokers))

## Exercise 6 — Use OR and a numeric threshold (Intermediate)

Select visits that happened on **Saturday or Sunday** and had a bill of at least **$25**. Show the day, total bill, tip, and party size. How many rows match? Use `|` for OR.

In [ ]:
"""Find larger bills from either weekend day."""
# The weekend condition is grouped before combining it with the bill threshold.
weekend_large_bills = tips.loc[
    (tips["day"].isin(["Sat", "Sun"])) & (tips["total_bill"] >= 25),
    ["day", "total_bill", "tip", "size"],
]
display(weekend_large_bills)
print("Matching records:", len(weekend_large_bills))

## Exercise 7 — Create a tip percentage column (Intermediate)

Create a new column named `tip_pct` by dividing `tip` by `total_bill` and multiplying by 100. Display the bill, tip, and new percentage for the first five visits. Then report the average percentage, rounded to one decimal place.

The percentage is a descriptive comparison of the recorded tip to the recorded bill; it is not a measure of service quality.

In [ ]:
"""Add and summarize tip as a percentage of the bill."""
# Divide row by row; total_bill is positive for these records.
tips["tip_pct"] = tips["tip"] / tips["total_bill"] * 100

# Display only the columns relevant to this calculation.
display(tips[["total_bill", "tip", "tip_pct"]].head())
print("Average tip percentage:", round(tips["tip_pct"].mean(), 1), "%")

## Exercise 8 — Create a category with a condition (Intermediate)

Add a `bill_size` column that labels bills of **$20 or more** as `Large` and bills under $20 as `Under $20`. Show a few rows and count how many records fall into each category.

In [ ]:
"""Label each bill using a $20 threshold and count the labels."""
# where() keeps the True value and substitutes the False value otherwise.
tips["bill_size"] = pd.Series(
    "Under $20",
    index=tips.index,
).where(tips["total_bill"] < 20, "Large")

display(tips[["total_bill", "bill_size"]].head())
print("Records by bill category:\n", tips["bill_size"].value_counts())

## Exercise 9 — Filter, select, and sort (Intermediate)

Among visits with a total bill greater than **$30**, find the five highest tip percentages. Show the day, meal period, total bill, tip, and `tip_pct`. Sort from highest to lowest percentage.

In [ ]:
"""Find the highest tip percentages among bills over $30."""
# Filter rows, choose explanatory columns, sort descending, then take five.
high_bill_tips = (
    tips.loc[
        tips["total_bill"] > 30,
        ["day", "time", "total_bill", "tip", "tip_pct"],
    ]
    .sort_values("tip_pct", ascending=False)
    .head(5)
)
display(high_bill_tips)

## Exercise 10 — Compare groups (Advanced)

Group the records by `day` and calculate the number of records, mean bill, mean tip, and mean tip percentage for each day. Sort the result by mean tip percentage from highest to lowest. Based on this sample, which day has the highest mean tip percentage? Keep in mind that an average based on different numbers of records may not tell the whole story.

In [ ]:
"""Compare bill and tip summaries across days of the week."""
# agg() computes several statistics for each day in one grouped result.
day_summary = (
    tips.groupby("day")
    .agg(
        records=("tip", "size"),
        mean_bill=("total_bill", "mean"),
        mean_tip=("tip", "mean"),
        mean_tip_pct=("tip_pct", "mean"),
    )
    .sort_values("mean_tip_pct", ascending=False)
)
display(day_summary.round(2))
print("Highest mean tip percentage in this sample:", day_summary["mean_tip_pct"].idxmax())

## Exercise 11 — Combine filtering, grouping, and ranking (Advanced challenge)

Use only `Dinner` records with a party size of **at least 3**. Group them by `smoker` and calculate the record count, average total bill, and average tip percentage. Sort by average tip percentage from highest to lowest. Which group has the larger average percentage in this filtered sample?

This is an association within a filtered dataset, not evidence that smoking status causes a change in tips.

In [ ]:
"""Summarize dinner visits for parties of three or more by smoker value."""
# Apply both row conditions before grouping the remaining records.
large_dinner_parties = tips.loc[
    (tips["time"] == "Dinner") & (tips["size"] >= 3)
]

# Report sample sizes alongside means so the comparison has context.
smoker_summary = (
    large_dinner_parties.groupby("smoker")
    .agg(
        records=("tip_pct", "size"),
        mean_bill=("total_bill", "mean"),
        mean_tip_pct=("tip_pct", "mean"),
    )
    .sort_values("mean_tip_pct", ascending=False)
)
display(smoker_summary.round(2))
print("Highest mean tip percentage in this filtered sample:", smoker_summary["mean_tip_pct"].idxmax())

## Reflection

1. Which pandas method helped you understand the columns and data types?
2. Why are parentheses important when combining conditions with `&` or `|`?
3. How does adding `tip_pct` make comparisons between different bill sizes easier?
4. Why should you check the record counts when comparing group averages?
5. What conclusions can and cannot be drawn from this restaurant sample?